In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils
import als

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    num_recs = 100


config = Config()

In [4]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, do_ordinal_encoder_transform=True, drop_pad=False, new_split=True)
X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
    config,
    X_train,
    X_val,
    X_val_seq,
    ordinal_encoder,
    return_only_val=True,
    truncate_val=False,
    return_just_splitted=True,
)
pad = ordinal_encoder.unknown_value

CPU times: user 17min 54s, sys: 2min 47s, total: 20min 41s
Wall time: 3min 46s


In [5]:
df_train_seqs = pl.read_parquet("train.parquet")

In [6]:
df_train_seqs = df_train_seqs.sample(frac=0.4, seed=1337)

In [7]:
%%time

data = list()
for i in tqdm(range(len(df_train_seqs))):
    row = df_train_seqs.row(i)
    row = [int(val) for val in row if val is not None][-config.max_len:]
    data.append(row)

  0%|          | 0/18312307 [00:00<?, ?it/s]

CPU times: user 1min 12s, sys: 4.93 s, total: 1min 17s
Wall time: 1min 17s


In [8]:
del df_train_seqs, row
gc.collect()

208

In [9]:
%%time

X_val_seq = X_val_seq.to_pandas()
X_val_seq[config.col_train_seq] = X_val_seq[config.col_aid].progress_apply(
    lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False)
)
X_val_seq = pl.from_pandas(X_val_seq)

  0%|          | 0/1671803 [00:00<?, ?it/s]

CPU times: user 8.3 s, sys: 507 ms, total: 8.81 s
Wall time: 8.77 s


In [10]:
# %%time

# X_train = X_train.to_pandas()
# X_train[config.col_train_seq] = X_train[config.col_aid].progress_apply(
#     lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False)
# )
# X_train = pl.from_pandas(X_train)

In [11]:
# X_train.head()

In [12]:
# train_val = pl.concat([X_train, X_val_seq[["user", "aid"]]])

In [13]:
# data = X_train.to_pandas()[config.col_train_seq].tolist()
# del X_train  # , train_val
# gc.collect()

In [14]:
%%time

model_als = als.train_als_model(data=data, n_items=pad + 1, iterations=20, factors=100, alpha=10.0, num_threads=0, use_gpu=True, calculate_training_loss=True)

  0%|          | 0/20 [00:00<?, ?it/s]

CPU times: user 5min 7s, sys: 5.17 s, total: 5min 12s
Wall time: 5min 13s


In [15]:
val_seq_data = X_val_seq.to_pandas()[config.col_train_seq].tolist()

In [16]:
%%time

recommender = als.AlsRecommender(implicit_model=model_als)
recs = recommender.recommend(val_seq_data, n_recs=config.num_recs)

100%|███████████████████████████████████████████| 17/17 [01:23<00:00,  4.90s/it]


CPU times: user 2min 42s, sys: 12.7 s, total: 2min 54s
Wall time: 2min 54s


In [17]:
df_idx = recs.to_pandas()[["index"]].drop_duplicates().reset_index(drop=True)
df_idx["user"] = X_val_seq.to_pandas()["user"].astype(int)
df_idx = pl.from_pandas(df_idx)
recs = recs.join(df_idx, on="index")
recs.head()

index,item_id,als_score,als_rank,user
i64,i64,f64,i64,i64
0,954978,0.999988,1,13732096
0,43341,0.00056,2,13732096
0,381616,0.000498,3,13732096
0,336606,0.000463,4,13732096
0,355826,0.000411,5,13732096


In [18]:
%%time

recs = recs.to_pandas()
recs['item_id'] = ordinal_encoder.inverse_transform(
        recs["item_id"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
recs = recs.dropna()
recs = pl.from_pandas(recs)

CPU times: user 23.5 s, sys: 15.3 s, total: 38.9 s
Wall time: 34.4 s


In [19]:
%%time

recs.write_parquet(config.cur_model_output_path + f'cur_als_df_val_preds_{config.num_recs}.parquet')

CPU times: user 16.4 s, sys: 5.65 s, total: 22.1 s
Wall time: 17.9 s


In [20]:
recs.head()

index,item_id,als_score,als_rank,user
i64,i64,f64,i64,i64
0,83986,0.00056,2,13732096
0,741952,0.000498,3,13732096
0,654388,0.000463,4,13732096
0,691809,0.000411,5,13732096
0,6851,0.000409,6,13732096


In [21]:
X_val_seq.head()

user,aid,train_seq
i64,list[i64],list[i64]
13732096,[954978],[954978]
14208832,[789634],[789634]
13909888,[145534],[145534]
13356736,[356965],[356965]
13375872,[837009],[837009]


In [22]:
X_val.filter(pl.col("user") == 14378816).head()

user,target,train_seq
i64,list[i64],list[i64]
14378816,"[257075, 1010867, 1578459]","[789219, 634920, ... 651794]"


In [23]:
recs = (
    recs.groupby(["index"])
    .agg(
        [
            pl.col("item_id").list(),
        ]
    )
    .sort("index")
    .to_pandas()
)

In [24]:
recs["user"] = X_val_seq.to_pandas()["user"]
recs.head()

,index,item_id,user
0,0,"[83986, 741952, 654388, 691809, 6851, 58326, 6...",13732096
1,1,"[836707, 1608431, 843000, 1798581, 473048, 925...",14208832
2,2,"[909449, 721034, 831165, 612920, 1203484, 1420...",13909888
3,3,"[95488, 485847, 337471, 1554258, 345807, 81928...",13356736
4,4,"[501077, 1199520, 767430, 984459, 77440, 14202...",13375872


In [25]:
%%time

recs["item_id"] = recs["item_id"].progress_apply(lambda x: utils.truncate_list(list(x), from_start=True, max_len=20))

  0%|          | 0/1671803 [00:00<?, ?it/s]

CPU times: user 16.2 s, sys: 2.77 s, total: 19 s
Wall time: 18.9 s


In [26]:
recs = recs.merge(X_val[["user", "target"]].to_pandas())

In [27]:
recs.head()

,index,item_id,user,target
0,2,"[909449, 721034, 831165, 612920, 1203484, 1420...",13909888,[1394429]
1,4,"[501077, 1199520, 767430, 984459, 77440, 14202...",13375872,[347486]
2,5,"[1257293, 1116621, 1534690, 801774, 485847, 15...",14093696,"[733503, 108251, 880557]"
3,6,"[337471, 95488, 597742, 345807, 1365988, 48584...",13378048,[1331520]
4,7,"[1116095, 42628, 159789, 765745, 538483, 85233...",13394432,"[817829, 1265509, 1700527]"


In [28]:
%%time

preds = recs["item_id"].tolist()
preds = [list(pred) for pred in preds]

targets = recs["target"].tolist()
targets = [list(target) for target in targets]

utils.recall_several_targets(preds, targets)

CPU times: user 3.33 s, sys: 123 ms, total: 3.45 s
Wall time: 3.47 s


0.06141846785034015

In [29]:
X_val_seq.head()

user,aid,train_seq
i64,list[i64],list[i64]
13732096,[954978],[954978]
14208832,[789634],[789634]
13909888,[145534],[145534]
13356736,[356965],[356965]
13375872,[837009],[837009]


In [30]:
recs

,index,item_id,user,target
0,2,"[909449, 721034, 831165, 612920, 1203484, 1420...",13909888,[1394429]
1,4,"[501077, 1199520, 767430, 984459, 77440, 14202...",13375872,[347486]
2,5,"[1257293, 1116621, 1534690, 801774, 485847, 15...",14093696,"[733503, 108251, 880557]"
3,6,"[337471, 95488, 597742, 345807, 1365988, 48584...",13378048,[1331520]
4,7,"[1116095, 42628, 159789, 765745, 538483, 85233...",13394432,"[817829, 1265509, 1700527]"
...,...,...,...,...
921699,1671788,"[1502122, 1157882, 11830, 1142000, 1383529, 23...",13796161,[624450]
921700,1671792,"[1502122, 1022566, 852331, 836852, 1778843, 54...",13150593,"[822597, 822597, 822597]"
921701,1671793,"[305158, 542343, 39846, 1192114, 316498, 12063...",13163009,[1645724]
921702,1671794,"[331708, 754412, 518425, 759436, 1853288, 1109...",13975681,[867018]


In [31]:
stop

NameError: name 'stop' is not defined

In [ ]:
X_val.to_pandas()[["user", "target"]]

In [ ]:
recs['target'] = 

In [ ]:
recs.filter(pl.col("index") == 1).head(20)

In [ ]:
stop

In [ ]:
%%time

rec_cooc = coocurence.CoocurenceRecommender(
    top_pairs=top_pairs, item_index=0, num_recs=config.num_recs, exclude_seen_items=False
)

data_to_predict = X_val_seq.to_pandas()[config.col_train_seq].tolist()
df_preds = rec_cooc.recommend(data_to_predict)

df_val_users = pl.from_pandas(X_val_seq.to_pandas()[[config.col_user]].reset_index())
df_val_preds = df_val_users.join(df_preds, on='index', how='inner')
df_val_preds = df_val_preds.with_columns(
    [
        pl.col('cooc_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_rank_0'),
    ]
)

df_val_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_df_val_preds_{config.num_recs}.parquet')

In [ ]:
del data_to_predict, df_preds, df_val_users, df_val_preds, X_val_seq, X_val
gc.collect()

In [ ]:
del top_pairs
gc.collect()

# TEST

In [ ]:
%%time

PATH_DATA = config.path_data
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_num_action] = df.groupby(config.col_user)[config.col_aid].cumcount()
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=True)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=True)
        .alias(config.col_n_session),
    ]
)

del df_train, df_test
gc.collect()

df_inter2inter = (
    df.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

list_inters = list(df_inter2inter.to_pandas()[config.col_aid])

del df_inter2inter, df
gc.collect()

In [ ]:
%%time

top_pairs = coocurence.train_cooc_model(data=list_inters, max_candidates=config.num_recs, normalize=True, filtering_rare_keys_trs=20)

In [ ]:
%%time

PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
df_test = df_test.sort(config.col_ts)
# df_test = df_test.to_pandas()
# df_test[config.col_aid] = ordinal_encoder.transform(
#     df_test[config.col_aid].values.reshape(-1, 1)
# ).astype(int)
# df_test = pl.from_pandas(df_test)
df_test = df_test.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

data_to_predict = df_test.to_pandas()[config.col_train_seq].tolist()
rec_cooc = coocurence.CoocurenceRecommender(
    top_pairs=top_pairs, item_index=0, num_recs=config.num_recs, exclude_seen_items=False
)
df_preds = rec_cooc.recommend(data_to_predict)

df_test_users = pl.from_pandas(df_test.to_pandas()[[config.col_user]].reset_index())
df_test_preds = df_test_users.join(df_preds, on='index', how='inner')
df_test_preds = df_test_preds.with_columns(
    [
        pl.col('cooc_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_rank_0'),
    ]
)

df_test_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_df_test_preds_{config.num_recs}.parquet')